In [5]:
!pip install xgboost imbalanced-learn


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\Bahissou TCHAGNAO\Desktop\CapCut\M2 DATA SCIENCE\FastAPI\.venv\Scripts\python.exe -m pip install --upgrade pip


In [11]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost", "imbalanced-learn"])

0

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import roc_auc_score, recall_score, f1_score

In [13]:
data= pd.read_csv("../data/data_clean.csv")

In [16]:
X =data.drop("statut_pret", axis=1)
y= data["statut_pret"]
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)


In [17]:
#  Encoder  les  variables catégorielles  en  utilisant  le  One-Hot Encoding
X = pd.get_dummies(X, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                    random_state=42, stratify=y)


In [29]:
# Preparation des  données pour la modélisation
scaler = StandardScaler()

#  Centré et  reduire  les  X  pour  la regression logistque
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


#===========================SMOTE===================================
#  sampling_strategy="auto"  veut  dire  que  SMOTE  va  générer  des  échantillons  
# synthétiques  pour  la  classe  minoritaire  jusqu'à ce que les classes soient équilibrées.
smote = SMOTE(random_state=42,sampling_strategy="auto")

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)


scaler_smote = StandardScaler()
X_train_smote_scaled = scaler_smote.fit_transform(X_train_smote)
X_test_smote_scaled = scaler_smote.transform(X_test)

#===========================xgboost
# Calcul  de  poids  pour  la  classe  minoritaire  et  la  classe  majoritaire
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"Poids de la classe minoritaire: {scale_weight:.2f}")

Poids de la classe minoritaire: 3.57


In [22]:
results = []
def eval_and_store(model_name, strategy_name, y_true, y_pred, y_proba):
    results.append({
        "Modèle": model_name,
        "Stratégie": strategy_name,
        "ROC-AUC": round(roc_auc_score(y_true, y_proba), 4),
        "Recall (Défaut)": round(recall_score(y_true, y_pred), 4),
        "F1-Score": round(f1_score(y_true, y_pred), 4)
    })

print("--- Entraînement et Évaluation des 9 configurations en cours... ---\n")

--- Entraînement et Évaluation des 9 configurations en cours... ---



In [ ]:
#  =================REGRESSION  LOGISTIQUE=========================

In [40]:
#Modele  de regression  logistique  classique

lr_base = LogisticRegression(random_state=42, max_iter=1000)
lr_base.fit(X_train_scaled, y_train)
eval_and_store("Régression Logistique", "Baseline",
                y_test, lr_base.predict(X_test_scaled), lr_base.predict_proba(X_test_scaled)[:, 1])

In [ ]:
# ======= Regression logistique ponderation des classes =========

lr_bal = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr_bal.fit(X_train_scaled, y_train)
eval_and_store("Régression Logistique", "Pondération", y_test, lr_bal.predict(X_test_scaled), lr_bal.predict_proba(X_test_scaled)[:, 1])

In [ ]:
# ==============regression  logistique  avec  SMOTE===========

lr_smote = LogisticRegression(random_state=42, max_iter=1000)
lr_smote.fit(X_train_smote_scaled, y_train_smote)
eval_and_store("Régression Logistique", "SMOTE", y_test, 
               lr_smote.predict(X_test_smote_scaled), lr_smote.predict_proba(X_test_smote_scaled)[:, 1])


In [ ]:
# =======RADOM FOREST CLASSIFIER   ==============

In [35]:
# ==============Random forest avec Baseline=============================

rf_base = RandomForestClassifier(random_state=42)
rf_base.fit(X_train, y_train)
eval_and_store("Random Forest", "Baseline", y_test, rf_base.predict(X_test), 
               rf_base.predict_proba(X_test)[:, 1])

In [33]:
# ==============Random forest avec Pondération=============================

rf_bal = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_bal.fit(X_train, y_train)
eval_and_store("Random Forest", "Pondération", y_test, rf_bal.predict(X_test),
                rf_bal.predict_proba(X_test)[:, 1])

In [34]:
# ==============Random forest avec SMOTE=============================

rf_smote = RandomForestClassifier(random_state=42)
rf_smote.fit(X_train_smote, y_train_smote)
eval_and_store("Random Forest", "SMOTE", y_test, rf_smote.predict(X_test), rf_smote.predict_proba(X_test)[:, 1])

In [ ]:
# =============XGBOOST CLASSIFIER=============================

In [ ]:
#==========xgboost avec baseline=======================

xgb_base = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_base.fit(X_train, y_train)
eval_and_store("XGBoost", "Baseline", y_test, xgb_base.predict(X_test), 
               xgb_base.predict_proba(X_test)[:, 1])

In [38]:
# =========xgboost avec pondération=======================

xgb_bal = XGBClassifier(scale_pos_weight=scale_weight, random_state=42, eval_metric='logloss')
xgb_bal.fit(X_train, y_train)
eval_and_store("XGBoost", "Pondération", y_test, xgb_bal.predict(X_test), xgb_bal.predict_proba(X_test)[:, 1])

In [37]:
#==================== xgboost avec SMOTE===================

xgb_smote = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_smote.fit(X_train_smote, y_train_smote)
eval_and_store("XGBoost", "SMOTE", y_test, xgb_smote.predict(X_test), xgb_smote.predict_proba(X_test)[:, 1])

In [ ]:
# =========TABLEAU COMPARATIF DES MODELES========================

In [43]:
df_benchmark = pd.DataFrame(results)
df_benchmark = df_benchmark.sort_values(by="ROC-AUC", ascending=False).reset_index(drop=True)

print(" TABLEAU COMPARATIF DES 9 CONFIGURATIONS :")
print(df_benchmark.to_string())

data.shape

 TABLEAU COMPARATIF DES 9 CONFIGURATIONS :
                   Modèle    Stratégie  ROC-AUC  Recall (Défaut)  F1-Score
0                 XGBoost     Baseline   0.9475           0.7489    0.8349
1                 XGBoost  Pondération   0.9443           0.7976    0.8067
2                 XGBoost        SMOTE   0.9428           0.7489    0.8336
3           Random Forest  Pondération   0.9279           0.7532    0.8153
4           Random Forest  Pondération   0.9279           0.7532    0.8153
5           Random Forest     Baseline   0.9274           0.7264    0.8260
6           Random Forest     Baseline   0.9274           0.7264    0.8260
7           Random Forest        SMOTE   0.9236           0.7433    0.8148
8   Régression Logistique     Baseline   0.8696           0.5776    0.6618
9   Régression Logistique  Pondération   0.8695           0.6206    0.6672
10  Régression Logistique     Baseline   0.8659           0.2786    0.4270
11  Régression Logistique     Baseline   0.8659          

(32411, 12)

In [53]:
import pandas as pd

# 1. Paramètres d'affichage Pandas pour éviter les troncatures
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.4f' % x) # Arrondit les probabilités à 4 chiffres

# 2. Création de votre tableau
tableau = pd.DataFrame({
    "vraie_classe": y_test,
    "classe_predite": predictions,
    "probabilite_classe_positive": probabilities,
    "score_risque_%": (probabilities * 100).round(2)
})

# 3. Utiliser display() au lieu de print() dans Jupyter pour un rendu HTML élégant
display(tableau.head(20))

,vraie_classe,classe_predite,probabilite_classe_positive,score_risque_%
21156,0,0,0.0190,1.9000
26850,1,0,0.1172,11.7200
26333,0,0,0.0669,6.6900
24056,0,0,0.0034,0.3400
16694,1,1,0.9829,98.2900
28804,0,0,0.3502,35.0200
28357,1,1,0.9612,96.1200
8426,0,0,0.1363,13.6300
29090,0,0,0.2768,27.6800
30187,0,0,0.0978,9.7800
